In [2]:
!rm -rf logs/ # clear logs
!rm -rf optimizer_output/

# Imports

In [3]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice again
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


In [4]:
import sympy as sp
import logging

from pathlib import Path

from symxplorer.spice_engine             import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools           import Nevergrad_Spice_Multi_Spec_Optimizer, Project_Setup

from symxplorer.logging import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

2025-09-30 07:27:53,943 - SymXplorer.optimizer - Using device: cpu and dtype: torch.float64
2025-09-30 07:27:53,957 - SymXplorer.jupyter - Spicelib_Wrapper imported successfully.


# Instantiations


In [5]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

07:27:53 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
07:27:54 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-09-30_07-27-53.log
07:27:54 - SymXplorer: [INFO] 🔧 spicelib logger set to 50


In [6]:
# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

07:27:54 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
07:27:54 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=25, random_seed=48
07:27:54 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
07:27:54 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
07:27:54 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
07:27:54 - SymXplorer.domains: [INFO] 	Number of target specs: 1
07:27:54 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=1e6, tolerance=10000.0, goal=OptimizationGoalType.EXACT, sim_type=SimType.AC, enable=True)
07:27:54 - SymXplorer.domains: [INFO] Project 'Tunable-TIA' initialized with simulator 'ngspice'
07:27:54 - SymXplorer.domains: [INFO] 	Workspace root: /foss/designs/eda/SymXplorer
07:27:54 - SymXplorer.domai

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(9.999999999999999e-06), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(9.999999999999999e-06), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(9.999999999999999e-06), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(9.999999999999999e-06), 'min_

In [7]:
# (2) Create the Spice Simulator Wrapper
wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename= PROJECT_SETUP.ws_root / PROJECT_SETUP.netlist,
    output_folder=PROJECT_SETUP.ws_root / PROJECT_SETUP.outdir,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

07:27:54 - SymXplorer.spicelib: [INFO] 📂 Creating output directory for the first time: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
07:27:54 - SymXplorer.spicelib: [INFO] --------------------------------------------------
07:27:54 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
07:27:54 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
07:27:54 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
07:27:54 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
07:27:54 - SymXplorer.spicelib: [INFO] --------------------------------------------------
07:27:54 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
07:27:54 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
07:27:54 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
07:27:54 - SymXplorer.spicelib: [INFO] Testbe

In [8]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Optimizer(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

# Sanity Check

In [9]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

07:27:54 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
07:27:54 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
07:27:55 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.log
07:27:55 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.raw
07:27:55 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
07:27:55 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Method Calls

In [10]:
circuit_optimizer.parameterize()

Dict(x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Log{Cl(0,6,b),exp=2.15},x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 10.000000000000002, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0}

In [11]:
circuit_optimizer.optimize()

07:27:55 - SymXplorer.optimizer: [INFO] Optimizer is set to CMA with budget = 25
Optimizing:  36%|███▌      | 9/25 [00:05<00:10,  1.56trial/s]07:28:01 - SymXplorer.spicelib: [ERROR] ❌ Variable fc not found in the raw file
07:28:01 - SymXplorer.optimizer: [CRITICAL] Target spec name 'fc' not found in performance array keys: ['fc']
07:28:01 - SymXplorer.optimizer: [WARNING] assigning large loss to the fc spec
Optimizing: 100%|██████████| 25/25 [00:17<00:00,  1.42trial/s]


[{'params': {'x_dut_nfet_w': 9.285528083527442,
   'x_dut_nfet_l': 44.040007658365084,
   'x_dut_cap_w': 40.33319781451455,
   'x_dut_cap_l': 53.245633899907325,
   'x_dut_res_s_l': 44.68527549637229,
   'x_dut_res_s_w': 50.41282611926594,
   'x_dut_res_3_l': 48.22058928690363,
   'x_dut_res_3_w': 47.77897205634083},
  'loss': np.float64(19994.629826532942),
  'metadata': {'fc': {'curr_val': np.float64(134.26335),
    'loss': np.float64(19994.629826532942)}}},
 {'params': {'x_dut_nfet_w': 6.53269617911447,
   'x_dut_nfet_l': 49.82444612599457,
   'x_dut_cap_w': 54.22319536532902,
   'x_dut_cap_l': 61.38370797024906,
   'x_dut_res_s_l': 41.48858798498957,
   'x_dut_res_s_w': 48.136185428444456,
   'x_dut_res_3_l': 49.462231286175914,
   'x_dut_res_3_w': 55.9567186367465},
  'loss': np.float64(16562.547824530055),
  'metadata': {'fc': {'curr_val': np.float64(89984.95000000001),
    'loss': np.float64(16562.547824530055)}}},
 {'params': {'x_dut_nfet_w': 7.443249676647304,
   'x_dut_nfet_l

In [12]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

07:28:13 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
07:28:13 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


In [13]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out

07:28:14 - SymXplorer.optimizer: [INFO] best loss: 1275.483575205


In [17]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

07:30:46 - SymXplorer.optimizer: [INFO] total loss: 1275.483575205
07:30:46 - SymXplorer.optimizer: [INFO] 	Spec 'fc': curr_val=747464.5, loss=1275.483575205
